# Sales Data Analysis
**Dataset:** `sales_data.csv`  |  **Records:** 100  |  **Period:** Jan – Apr 2024

---


In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("sales_data.csv", parse_dates=["Date"])
print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

Loaded 100 rows × 7 columns


,Date,Product,Quantity,Price,Customer_ID,Region,Total_Sales
0,2024-01-01,Phone,7,37300,CUST001,East,261100
1,2024-01-02,Headphones,4,15406,CUST002,North,61624
2,2024-01-03,Phone,2,21746,CUST003,West,43492
3,2024-01-04,Headphones,1,30895,CUST004,East,30895
4,2024-01-05,Laptop,8,39835,CUST005,North,318680


In [2]:
print("Shape     :", df.shape)
print("Columns   :", df.columns.tolist())
print()
print("Data Types")
print(df.dtypes)

Shape     : (100, 7)
Columns   : ['Date', 'Product', 'Quantity', 'Price', 'Customer_ID', 'Region', 'Total_Sales']

Data Types
Date           datetime64[ns]
Product                object
Quantity                int64
Price                   int64
Customer_ID            object
Region                 object
Total_Sales             int64
dtype: object


In [3]:
df.describe()

,Date,Quantity,Price,Total_Sales
count,100,100.000000,100.000000,100.000000
mean,2024-02-19 12:00:00,4.780000,25808.510000,123650.480000
min,2024-01-01 00:00:00,1.000000,1308.000000,6540.000000
25%,2024-01-25 18:00:00,2.750000,14965.250000,39517.500000
50%,2024-02-19 12:00:00,5.000000,24192.000000,97955.500000
75%,2024-03-15 06:00:00,7.000000,38682.250000,175792.500000
max,2024-04-09 00:00:00,9.000000,49930.000000,373932.000000
std,NaN,2.588163,13917.630242,100161.085275


In [4]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

Missing values per column:
Date           0
Product        0
Quantity       0
Price          0
Customer_ID    0
Region         0
Total_Sales    0
dtype: int64

Duplicate rows: 0


In [5]:
before = len(df)
df = df.drop_duplicates().dropna().reset_index(drop=True)
df["Month"] = df["Date"].dt.to_period("M")
after = len(df)
print(f"Rows before : {before}")
print(f"Rows after  : {after}")
print(f"Removed     : {before - after}")
print("\nData is clean — no duplicates or missing values found.")

Rows before : 100
Rows after  : 100
Removed     : 0

Data is clean — no duplicates or missing values found.


In [7]:
total_revenue   = df["Total_Sales"].sum()
total_quantity  = df["Quantity"].sum()
avg_order_value = df["Total_Sales"].mean()

print(f"Total Revenue    : ₹{total_revenue:,.0f}")
print(f"Total Units Sold : {total_quantity:,}")
print(f"Avg Order Value  : ₹{avg_order_value:,.2f}")

Total Revenue    : ₹12,365,048
Total Units Sold : 478
Avg Order Value  : ₹123,650.48


In [8]:
by_product = (
    df.groupby("Product")
    .agg(
        Revenue     = ("Total_Sales", "sum"),
        Units_Sold  = ("Quantity",    "sum"),
        Avg_Price   = ("Price",       "mean"),
        Orders      = ("Customer_ID", "count"),
    )
    .sort_values("Revenue", ascending=False)
)

by_product["Revenue_Share_%"] = (by_product["Revenue"] / total_revenue * 100).round(1)
by_product

,Revenue,Units_Sold,Avg_Price,Orders,Revenue_Share_%
Product,,,,,
Laptop,3889210,136,27651.500000,24,31.5
Tablet,2884340,127,24177.230769,26,23.3
Phone,2859394,101,27379.000000,20,23.1
Headphones,1384033,48,28692.133333,15,11.2
Monitor,1348071,66,20709.666667,15,10.9


In [9]:
best_product = by_product["Revenue"].idxmax()
best_revenue = by_product.loc[best_product, "Revenue"]
print(f"Best-Selling Product : {best_product}")
print(f"Revenue Contribution : ₹{best_revenue:,.0f}")

Best-Selling Product : Laptop
Revenue Contribution : ₹3,889,210


In [10]:
by_region = (
    df.groupby("Region")["Total_Sales"]
    .sum()
    .sort_values(ascending=False)
    .rename("Revenue")
    .to_frame()
)
by_region["Revenue_Share_%"] = (by_region["Revenue"] / total_revenue * 100).round(1)
by_region

,Revenue,Revenue_Share_%
Region,,
North,3983635,32.2
South,3737852,30.2
East,2519639,20.4
West,2123922,17.2


In [11]:
by_month = (
    df.groupby("Month")["Total_Sales"]
    .sum()
    .rename("Revenue")
    .to_frame()
)
by_month["Revenue_Share_%"] = (by_month["Revenue"] / total_revenue * 100).round(1)
by_month

,Revenue,Revenue_Share_%
Month,,
2024-01,4120524,33.3
2024-02,2656050,21.5
2024-03,4485006,36.3
2024-04,1103468,8.9


In [15]:
sep = "=" * 50
print(sep)
print("         SALES ANALYSIS REPORT")
print(sep)

print("\n  OVERVIEW")
print(f"  Total Revenue    : ₹{total_revenue:,.0f}")
print(f"  Total Units Sold : {total_quantity:,}")
print(f"  Avg Order Value  : ₹{avg_order_value:,.2f}")

print(f"\n  Best-Selling Product : {best_product}")
print(f"  Top Region           : {by_region['Revenue'].idxmax()}")

print("\n  Revenue by Product")
for product, row in by_product.iterrows():
    bar = "█" * int(row["Revenue"] / 200_000)
    print(f"  {product:<12} ₹{int(row['Revenue']):>10,}  {row['Revenue_Share_%']:5.1f}%  {bar}")

print("\n  Revenue by Region")
for region, row in by_region.iterrows():
    print(f"  {region:<8} ₹{int(row['Revenue']):>10,}  {row['Revenue_Share_%']:5.1f}%")

print("\n  Revenue by Month")
for month, row in by_month.iterrows():
    print(f"  {str(month):<10} ₹{int(row['Revenue']):>10,}  {row['Revenue_Share_%']:5.1f}%")

print(f"\n{sep}")

print("\n  KEY INSIGHTS")
print(f"  1. Laptop dominates with 31.5% of total revenue.")
print(f"  2. North region leads sales at 32.2% of total.")
print(f"  3. March 2024 was the strongest month (₹4.48M).")
print(f"  4. Headphones & Monitor together account for only 22% — potential growth areas.")
print(f"  5. April shows a sharp drop — likely incomplete data or seasonal dip.")

         SALES ANALYSIS REPORT

  OVERVIEW
  Total Revenue    : ₹12,365,048
  Total Units Sold : 478
  Avg Order Value  : ₹123,650.48

  Best-Selling Product : Laptop
  Top Region           : North

  Revenue by Product
  Laptop       ₹ 3,889,210   31.5%  ███████████████████
  Tablet       ₹ 2,884,340   23.3%  ██████████████
  Phone        ₹ 2,859,394   23.1%  ██████████████
  Headphones   ₹ 1,384,033   11.2%  ██████
  Monitor      ₹ 1,348,071   10.9%  ██████

  Revenue by Region
  North    ₹ 3,983,635   32.2%
  South    ₹ 3,737,852   30.2%
  East     ₹ 2,519,639   20.4%
  West     ₹ 2,123,922   17.2%

  Revenue by Month
  2024-01    ₹ 4,120,524   33.3%
  2024-02    ₹ 2,656,050   21.5%
  2024-03    ₹ 4,485,006   36.3%
  2024-04    ₹ 1,103,468    8.9%


  KEY INSIGHTS
  1. Laptop dominates with 31.5% of total revenue.
  2. North region leads sales at 32.2% of total.
  3. March 2024 was the strongest month (₹4.48M).
  4. Headphones & Monitor together account for only 22% — potential grow

---
*Analysis complete. See `analysis_report.md` for the full written report.*
